Create a partitioned managed table and do zordering on top of it

In [0]:
%sql
drop table if exists nyc_taxi;
create table nyc_taxi (
   vendor_id	string,
pickup_datetime	timestamp,
dropoff_datetime	timestamp,
passenger_count	int,
trip_distance	double,
pickup_longitude	double,
pickup_latitude	double,
rate_code_id	int,
store_and_fwd_flag	string,
dropoff_longitude	double,
dropoff_latitude	double,
payment_type	string,
fare_amount	double,
extra	double,
mta_tax	double,
tip_amount	double,
tolls_amount	double,
total_amount	double
) using delta
partitioned by (vendor_id)
tblproperties (
delta.autoOptimize.optimizeWrite = false,
delta.autoOptimize.autoCompact = false
)

In [0]:
file_path = 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow'
all_files = [f.path for f in dbutils.fs.ls(file_path) if f.path.endswith('parquet')]

files10 = all_files[:5]
df = spark.read.parquet(*files10)
df1 = df.repartition(200)

df1.write.format('delta').mode('overwrite').saveAsTable("nyc_taxi")

In [0]:
%sql
describe history nyc_taxi

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-08-31T15:33:37.000Z,147836707444603,anooptu@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [""vendor_id""], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.autoOptimize.autoCompact"":""false"",""delta.enableDeletionVectors"":""true"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-fa6faed8-533c-4cfe-934d-1690b8d3aec7"",""delta.autoOptimize.optimizeWrite"":""false"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-a9c081bc-d20a-4156-a337-40db703d64f8""}, statsOnLoad -> true)",null,List(3322194627801979),e47c9068-f25b-46c4-864a-c28449b36a1d,0831-144423-i1ydcey7-v2n,0,WriteSerializable,false,"Map(numFiles -> 601, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 45142478, numOutputBytes -> 1270180807)",null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13
0,2026-08-31T15:32:24.000Z,147836707444603,anooptu@gmail.com,CREATE TABLE,"Map(partitionBy -> [""vendor_id""], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.autoOptimize.autoCompact"":""false"",""delta.enableDeletionVectors"":""true"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-fa6faed8-533c-4cfe-934d-1690b8d3aec7"",""delta.autoOptimize.optimizeWrite"":""false"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-a9c081bc-d20a-4156-a337-40db703d64f8""}, statsOnLoad -> false)",null,List(3322194627801979),57958bc1-0d1e-42ff-baeb-1a9e020eb25c,0831-144423-i1ydcey7-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13


In [0]:
%sql
select min(trip_distance) ,max(trip_distance) , _metadata.file_name from nyc_taxi
group by _metadata.file_name order by  min(trip_distance);


min(trip_distance),max(trip_distance),file_name
0.0,88.0,part-00010-1bd924e7-f04e-4b96-8aa1-0b07c19522b4.c000.zstd.parquet
0.0,202.0,part-00046-602d8af8-22a4-4fc6-b826-1556433d1d89.c000.zstd.parquet
0.0,54.0,part-00082-a966b293-4cf2-4bdd-9b23-3dae522f104a.c000.zstd.parquet
0.0,813.1,part-00077-11cfc973-f640-4b42-8108-2ae46c470960.c000.zstd.parquet
0.0,183.1,part-00146-50278817-92d2-4501-a085-a34b7ee85ff6.c000.zstd.parquet
0.0,48.3,part-00169-7a2db556-90e6-4cb8-a7d7-48db6b6c9e0a.c000.zstd.parquet
0.0,205.0,part-00060-fef6060b-a1f9-46e5-b0b8-f9f3a50ec5dd.c000.zstd.parquet
0.0,60.8,part-00120-ec76b06b-c558-4524-9283-4f3bff03f416.c000.zstd.parquet
0.0,504.1,part-00069-8485990b-5a89-4239-8528-41de14cda835.c000.zstd.parquet
0.0,177.7,part-00061-17fe5f6c-826a-4fc2-8f01-7835d951be51.c000.zstd.parquet


In [0]:
%sql
select count(*) from nyc_taxi where trip_distance > 100;

count(*)
158


Files & partitions		
- Files read	115	
- Files pruned	486	
- Partitions read	2	

On top of partitioning, do a zordering

In [0]:
%sql
optimize nyc_taxi zorder by trip_distance

path,metrics
abfss://unity-catalog-storage@dbstoragekf4btudtg6f2k.dfs.core.windows.net/7405613815916659/__unitystorage/catalogs/c81fd24d-dda0-4fd3-afe6-9159d22a76fa/tables/c813b0fc-3dcb-40f5-b1bc-31a5862d98bf,"List(18, 600, List(42899058, 74425415, 5.8341163722222224E7, 18, 1050140947), List(294703, 3387488, 2116960.15, 600, 1270176090), 4, List(minCubeSize(107374182400), List(0, 0), List(601, 1270180807), 0, List(600, 1270176090), 3, null), null, 0, 1, 601, 1, false, 0, 0, 1788190623675, 1788190641933, 16, 3, null, List(0, 0), null, 18, 18, 77073, 0, null, null, 0)"


Files inside Each partition will be z ordered separatly, maintaining the partitioning

In [0]:
%sql
select min(trip_distance) ,max(trip_distance) , _metadata.file_name from nyc_taxi
group by _metadata.file_name order by  min(trip_distance);


min(trip_distance),max(trip_distance),file_name
0.0,0.7,part-00000-2e45e822-fcd9-47f7-8abf-20280e716026.c000.zstd.parquet
0.0,0.82,part-00010-23cee408-e729-4c49-8d8e-b85e4247a12c.c000.zstd.parquet
0.0,48.7,part-00009-5b87d813-5e41-4180-8d4f-914496134bc7.c000.zstd.parquet
0.4,0.4,part-00091-c666eee2-7a70-4f7d-afa2-ed4fc5cf593b.c000.zstd.parquet
0.7,1.0,part-00001-2d1f001c-f72a-4c3e-946d-7ddf31607f7a.c000.zstd.parquet
0.82,1.14,part-00011-5888c71f-8379-4b21-a32e-c0560583e273.c000.zstd.parquet
1.0,1.4,part-00002-742e82e2-495e-4b80-859e-4bc7703cea42.c000.zstd.parquet
1.14,1.51,part-00012-268b088d-ff27-41c8-9180-a62a457809e3.c000.zstd.parquet
1.4,1.8,part-00003-a65fcb6b-576a-4a81-b2fb-4bec84bdd965.c000.zstd.parquet
1.51,2.06,part-00013-4336ce6f-8b3c-473c-8b92-95e097e2761f.c000.zstd.parquet


Some ranges are overlapping still, as they are in different partitions. But query runs faster

In [0]:
%sql
select count(*) from nyc_taxi where trip_distance > 100;

count(*)
158


Files & partitions		
- Files read	2	
- Files pruned	17	
- Partitions read	2

Now drop and recreate managed table  with liquid clustering

In [0]:
%sql
drop table if exists nyc_taxi;
create table nyc_taxi (
   vendor_id	string,
pickup_datetime	timestamp,
dropoff_datetime	timestamp,
passenger_count	int,
trip_distance	double,
pickup_longitude	double,
pickup_latitude	double,
rate_code_id	int,
store_and_fwd_flag	string,
dropoff_longitude	double,
dropoff_latitude	double,
payment_type	string,
fare_amount	double,
extra	double,
mta_tax	double,
tip_amount	double,
tolls_amount	double,
total_amount	double
) using delta
cluster by (vendor_id ,trip_distance)
tblproperties (
delta.autoOptimize.optimizeWrite = false,
delta.autoOptimize.autoCompact = false
)

In [0]:
file_path = 'dbfs:/databricks-datasets/nyctaxi/tables/nyctaxi_yellow'
all_files = [f.path for f in dbutils.fs.ls(file_path) if f.path.endswith('parquet')]

files10 = all_files[:5]
df = spark.read.parquet(*files10)
df1 = df.repartition(200)

df1.write.format('delta').mode('overwrite').saveAsTable("nyc_taxi")

In [0]:
%sql
describe detail nyc_taxi;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,3ef9a725-0922-43c3-95ae-d50bfe0c0e08,useastws.default.nyc_taxi,null,abfss://unity-catalog-storage@dbstoragekf4btudtg6f2k.dfs.core.windows.net/7405613815916659/__unitystorage/catalogs/c81fd24d-dda0-4fd3-afe6-9159d22a76fa/tables/b944fe99-a2a8-4c00-b136-b4f77a27b069,2026-08-31T15:43:12.508Z,2026-08-31T15:44:08.000Z,List(),"List(vendor_id, trip_distance)",15,1021546521,"Map(delta.parquet.compression.codec -> zstd, delta.autoOptimize.autoCompact -> false, delta.enableDeletionVectors -> true, databricks.delta.expressionStats.selectedColumns -> upper(vendor_id),lower(vendor_id), delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, delta.autoOptimize.optimizeWrite -> false, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-3d18113e-3c99-4ca7-ba36-398573d0b1c0, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-79962629-fbf3-4b9c-895b-2133944efa74)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
select count(*) from nyc_taxi where trip_distance > 100;

count(*)
158


After liquid clustering
IO		
- Rows returned	1	
- Rows read	158	
- Bytes read	2.56	MB
- Bytes pruned	774.07	MB

Files & partitions		
- Files read	2	
- Files pruned	13	
- Partitions read	0	

In [0]:
%sql
select min(trip_distance) ,max(trip_distance) , _metadata.file_name from nyc_taxi
group by _metadata.file_name order by  min(trip_distance);


min(trip_distance),max(trip_distance),file_name
0.0,0.7,part-00006-2c6aa7d9-057e-47f8-a4eb-5eb2509eb010.c000.zstd.parquet
0.0,0.9289999999999999,part-00000-70025567-a91f-4e31-8ea8-47fac0b0bf13.c000.zstd.parquet
0.0,1.7,part-00004-6dbd7226-a143-42c6-a841-8693f81709fd.c000.zstd.parquet
0.8,1.1,part-00013-9b4c09a9-67b0-4b5e-9b8f-864745f333f3.c000.zstd.parquet
0.93,1.407,part-00005-6bde6ad8-72b4-47ce-a216-902d07696e4b.c000.zstd.parquet
1.2,1.4,part-00009-f258f08e-9838-4137-b578-12e2b24ae05b.c000.zstd.parquet
1.41,2.01,part-00001-ab4f4cbc-48d1-439b-80b2-bd4c3d5e73be.c000.zstd.parquet
1.5,1.7,part-00012-d911d121-14d1-426f-b987-47056b990e10.c000.zstd.parquet
1.8,48.7,part-00002-fe6ebb2c-0e77-4f75-821f-6365391f0dc2.c000.zstd.parquet
1.8,2.7,part-00003-3edb5748-d69f-43ab-8e80-a510af7c0955.c000.zstd.parquet


Just 15 files in total and also arranged as ranges of trip distance

In [0]:
%sql
describe history nyc_taxi


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-08-31T15:44:08.000Z,147836707444603,anooptu@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [""vendor_id"",""trip_distance""], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.autoOptimize.autoCompact"":""false"",""delta.enableDeletionVectors"":""true"",""databricks.delta.expressionStats.selectedColumns"":""upper(vendor_id),lower(vendor_id)"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-79962629-fbf3-4b9c-895b-2133944efa74"",""delta.autoOptimize.optimizeWrite"":""false"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-3d18113e-3c99-4ca7-ba36-398573d0b1c0""}, statsOnLoad -> true, clusteringOnWriteStatus -> late-stage clustering triggered)",null,List(3322194627801979),d18673ae-f0e3-4dea-a40e-1944dd3db8bf,0831-144423-i1ydcey7-v2n,0,WriteSerializable,false,"Map(numFiles -> 15, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 45142478, numOutputBytes -> 1021546521)",null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13
0,2026-08-31T15:43:13.000Z,147836707444603,anooptu@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [""vendor_id"",""trip_distance""], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.autoOptimize.autoCompact"":""false"",""delta.enableDeletionVectors"":""true"",""databricks.delta.expressionStats.selectedColumns"":""upper(vendor_id),lower(vendor_id)"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-79962629-fbf3-4b9c-895b-2133944efa74"",""delta.autoOptimize.optimizeWrite"":""false"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-3d18113e-3c99-4ca7-ba36-398573d0b1c0""}, statsOnLoad -> false)",null,List(3322194627801979),ac73e6fb-a7ce-49bf-849e-d8f4a3f7bf29,0831-144423-i1ydcey7-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13


In [0]:
%sql
optimize nyc_taxi

path,metrics
abfss://unity-catalog-storage@dbstoragekf4btudtg6f2k.dfs.core.windows.net/7405613815916659/__unitystorage/catalogs/c81fd24d-dda0-4fd3-afe6-9159d22a76fa/tables/b944fe99-a2a8-4c00-b136-b4f77a27b069,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 15, 0, false, 0, 0, 1788191605177, 1788191611003, 8, 0, null, List(0, 0), null, 18, 18, 0, 0, List(1021546521, true, false, false, null, null, null, null, 0, 12, 903818692, 903818692, 3, 117727829, 117727829, null, log, 16777216, 67108864, 4, 0, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(207, 86, 0, 0, 0, 2385), 2, 1, 5, sizeAware, hierarchicalSplitStrategy, null, false, 0, null, false, 0, 0, 0, null, null, null), null, 0)"
abfss://unity-catalog-storage@dbstoragekf4btudtg6f2k.dfs.core.windows.net/7405613815916659/__unitystorage/catalogs/c81fd24d-dda0-4fd3-afe6-9159d22a76fa/tables/b944fe99-a2a8-4c00-b136-b4f77a27b069,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 15, 15, true, 0, 0, 1788191611034, 1788191613333, 8, 0, null, List(0, 0), null, 18, 18, 0, 0, List(1021546521, false, false, false, null, null, null, post-optimize-compaction, 0, 0, 0, 0, 0, 0, 0, null, null, 33554432, 67108864, 0, 0, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(0, 0, 798, 0, 0, 0), 15, 1, 1, null, null, null, false, 0, null, false, 0, 0, 0, null, null, null), null, 0)"


In [0]:
%sql
describe history nyc_taxi


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-08-31T15:53:29.000Z,147836707444603,anooptu@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> false, clusterBy -> [""vendor_id"",""trip_distance""], isFull -> false, zOrderBy -> [], batchId -> -1)",null,List(3322194627801979),b2bc7b65-752c-4ff2-9840-3a9ef3d8dd84,0831-144423-i1ydcey7-v2n,1,SnapshotIsolation,true,Map(conflictDetectionTimeMs -> 41),null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13
2,2026-08-31T15:53:28.000Z,147836707444603,anooptu@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> false, clusterBy -> [""vendor_id"",""trip_distance""], isFull -> false, zOrderBy -> [], batchId -> -1)",null,List(3322194627801979),b2bc7b65-752c-4ff2-9840-3a9ef3d8dd84,0831-144423-i1ydcey7-v2n,1,WriteSerializable,true,Map(),null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13
1,2026-08-31T15:44:08.000Z,147836707444603,anooptu@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [""vendor_id"",""trip_distance""], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.autoOptimize.autoCompact"":""false"",""delta.enableDeletionVectors"":""true"",""databricks.delta.expressionStats.selectedColumns"":""upper(vendor_id),lower(vendor_id)"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-79962629-fbf3-4b9c-895b-2133944efa74"",""delta.autoOptimize.optimizeWrite"":""false"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-3d18113e-3c99-4ca7-ba36-398573d0b1c0""}, statsOnLoad -> true, clusteringOnWriteStatus -> late-stage clustering triggered)",null,List(3322194627801979),d18673ae-f0e3-4dea-a40e-1944dd3db8bf,0831-144423-i1ydcey7-v2n,0,WriteSerializable,false,"Map(numFiles -> 15, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 45142478, numOutputBytes -> 1021546521)",null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13
0,2026-08-31T15:43:13.000Z,147836707444603,anooptu@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [""vendor_id"",""trip_distance""], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.autoOptimize.autoCompact"":""false"",""delta.enableDeletionVectors"":""true"",""databricks.delta.expressionStats.selectedColumns"":""upper(vendor_id),lower(vendor_id)"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-79962629-fbf3-4b9c-895b-2133944efa74"",""delta.autoOptimize.optimizeWrite"":""false"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-3d18113e-3c99-4ca7-ba36-398573d0b1c0""}, statsOnLoad -> false)",null,List(3322194627801979),ac73e6fb-a7ce-49bf-849e-d8f4a3f7bf29,0831-144423-i1ydcey7-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13


In [0]:
%sql
select min(trip_distance) ,max(trip_distance) , _metadata.file_name from nyc_taxi
group by _metadata.file_name order by  min(trip_distance);


min(trip_distance),max(trip_distance),file_name
0.0,0.9289999999999999,part-00000-70025567-a91f-4e31-8ea8-47fac0b0bf13.c000.zstd.parquet
0.0,0.7,part-00006-2c6aa7d9-057e-47f8-a4eb-5eb2509eb010.c000.zstd.parquet
0.0,1.7,part-00004-6dbd7226-a143-42c6-a841-8693f81709fd.c000.zstd.parquet
0.8,1.1,part-00013-9b4c09a9-67b0-4b5e-9b8f-864745f333f3.c000.zstd.parquet
0.93,1.407,part-00005-6bde6ad8-72b4-47ce-a216-902d07696e4b.c000.zstd.parquet
1.2,1.4,part-00009-f258f08e-9838-4137-b578-12e2b24ae05b.c000.zstd.parquet
1.41,2.01,part-00001-ab4f4cbc-48d1-439b-80b2-bd4c3d5e73be.c000.zstd.parquet
1.5,1.7,part-00012-d911d121-14d1-426f-b987-47056b990e10.c000.zstd.parquet
1.8,2.7,part-00003-3edb5748-d69f-43ab-8e80-a510af7c0955.c000.zstd.parquet
1.8,48.7,part-00002-fe6ebb2c-0e77-4f75-821f-6365391f0dc2.c000.zstd.parquet


Files were already optimized

Auto mode allowed on managed table

In [0]:
%sql
alter table nyc_taxi cluster by auto;

In [0]:
%sql
optimize nyc_taxi

path,metrics
abfss://unity-catalog-storage@dbstoragekf4btudtg6f2k.dfs.core.windows.net/7405613815916659/__unitystorage/catalogs/c81fd24d-dda0-4fd3-afe6-9159d22a76fa/tables/77329acf-efe6-4bfd-a3dd-182733a746a3,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 15, 0, false, 0, 0, 1788194527890, 1788194537608, 32, 0, null, List(0, 0), null, 18, 18, 0, 0, List(1019826902, true, false, false, null, null, null, null, 0, 10, 805377510, 805377510, 5, 214449392, 214449392, null, log, 16777216, 67108864, 4, 0, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(276, 10, 0, 0, 0, 5215), 2, 1, 5, sizeAware, hierarchicalSplitStrategy, null, false, 0, null, false, 0, 0, 0, null, null, null), null, 0)"
abfss://unity-catalog-storage@dbstoragekf4btudtg6f2k.dfs.core.windows.net/7405613815916659/__unitystorage/catalogs/c81fd24d-dda0-4fd3-afe6-9159d22a76fa/tables/77329acf-efe6-4bfd-a3dd-182733a746a3,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 15, 15, true, 0, 0, 1788194537631, 1788194540723, 32, 0, null, List(0, 0), null, 18, 18, 0, 0, List(1019826902, false, false, false, null, null, null, post-optimize-compaction, 0, 0, 0, 0, 0, 0, 0, null, null, 33554432, 67108864, 0, 0, 0, 0, null, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, List(0, 0, 1178, 0, 0, 0), 15, 1, 1, null, null, null, false, 0, null, false, 0, 0, 0, null, null, null), null, 0)"


In [0]:
%sql
describe detail nyc_taxi;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,85ed7162-f398-49bd-98e6-816f1f67b672,useastws.default.nyc_taxi,null,abfss://unity-catalog-storage@dbstoragekf4btudtg6f2k.dfs.core.windows.net/7405613815916659/__unitystorage/catalogs/c81fd24d-dda0-4fd3-afe6-9159d22a76fa/tables/77329acf-efe6-4bfd-a3dd-182733a746a3,2026-08-31T16:39:26.549Z,2026-08-31T16:42:15.000Z,List(),"List(vendor_id, trip_distance)",15,1019826902,"Map(delta.parquet.compression.codec -> zstd, delta.autoOptimize.autoCompact -> false, delta.enableDeletionVectors -> true, databricks.delta.expressionStats.selectedColumns -> upper(vendor_id),lower(vendor_id), delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, delta.autoOptimize.optimizeWrite -> false, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-59e5dfca-7228-4250-8bff-95ffb5b36833, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-a8a1d240-a180-480c-bf69-f6c1fc024a28)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",true
